# 01 - Raw Exploration

Open-Meteo feature coverage, missingness, and AQI distribution audit.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
load_dotenv(str(ROOT / ".env"), override=False)

from src.data.fetch_openmeteo import AIR_QUALITY_HOURLY, WEATHER_HOURLY
from src.utils.mongo_client import get_database

sns.set_theme(style="whitegrid")

db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))

if data.empty:
    raise ValueError("aqi_features_rawalpindi is empty")

if "_id" in data.columns:
    data = data.drop(columns=["_id"])

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

data.head()

In [ ]:
missing_weather_columns = sorted(set(WEATHER_HOURLY) - set(data.columns))
missing_air_columns = sorted(set(AIR_QUALITY_HOURLY) - set(data.columns))
required_fields = ["timestamp", "european_aqi", "pm2_5", "pm10"]

coverage = pd.DataFrame(
    {
        "dataset": ["weather", "air_quality", "required_fields"],
        "expected": [len(WEATHER_HOURLY), len(AIR_QUALITY_HOURLY), len(required_fields)],
        "present": [
            len(WEATHER_HOURLY) - len(missing_weather_columns),
            len(AIR_QUALITY_HOURLY) - len(missing_air_columns),
            sum(field in data.columns for field in required_fields),
        ],
        "missing": [
            len(missing_weather_columns),
            len(missing_air_columns),
            sum(field not in data.columns for field in required_fields),
        ],
    }
)

display(coverage)
print("Missing weather columns:", missing_weather_columns or ["None"])
print("Missing air-quality columns:", missing_air_columns or ["None"])

In [ ]:
missingness = data.isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame()
top_missing = missingness.head(20)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(data["european_aqi"].dropna(), bins=40, ax=axes[0], color="#0f766e")
axes[0].set_title("European AQI")
sns.histplot(data["pm2_5"].dropna(), bins=40, ax=axes[1], color="#2563eb")
axes[1].set_title("PM2.5")
sns.histplot(data["pm10"].dropna(), bins=40, ax=axes[2], color="#f97316")
axes[2].set_title("PM10")
plt.tight_layout()

display(top_missing)

In [ ]:
hourly_index = pd.date_range(data["timestamp"].min(), data["timestamp"].max(), freq="H", tz="UTC")
gap_hours = data["timestamp"].diff().dt.total_seconds().div(3600)
gaps = gap_hours[gap_hours > 1].dropna()

monthly = (
    data.assign(month=data["timestamp"].dt.to_period("M").dt.to_timestamp())
    .groupby("month")
    .agg(
        rows=("timestamp", "size"),
        mean_aqi=("european_aqi", "mean"),
        pm25_mean=("pm2_5", "mean"),
        pm10_mean=("pm10", "mean"),
    )
)

fig, ax = plt.subplots(figsize=(11, 4))
monthly["mean_aqi"].plot(kind="line", marker="o", ax=ax, color="#f97316")
ax.set_title("Monthly Average European AQI")
ax.set_ylabel("AQI")
plt.tight_layout()

display(
    pd.DataFrame(
        {
            "metric": ["rows", "expected_hours", "hourly_coverage", "gap_count"],
            "value": [len(data), len(hourly_index), len(data) / len(hourly_index), len(gaps)],
        }
    )
)
display(monthly)